In [ ]:
import pandas as pd

ident = pd.read_csv("../data/raw/GSE133344_filtered_cell_identities.csv.gz")
print(ident.shape)
print(ident.head())
print(ident.columns.tolist())

# col = cell_barcode is the key for joining with adata.obs (metadata) later


In [ ]:
import scanpy as sc

adata = sc.read_mtx("../data/raw/GSE133344_filtered_matrix.mtx.gz").T
# the barcodes file was probably made by cellranger aggr, after indiv cellranger runs per lane.  barcodes already have laneID appended
barcodes = pd.read_csv("../data/raw/GSE133344_filtered_barcodes.tsv.gz", header=None)[0]
genes = pd.read_csv("../data/raw/GSE133344_filtered_genes.tsv.gz", header=None, sep="\t")

# adata.obs is metadata, indexed by barcode-laneID.
# adata.var is metadata for genes.  when we load genes[1].values into indices for var, it doesn't yet have any columns

adata.obs_names = barcodes.values
adata.var_names = genes[1].values  # gene symbol column; check genes.head() if this looks wrong
adata.var_names_make_unique()

print(adata)
print(adata.obs_names[:5].tolist())

In [ ]:
# ident was loaded in in the first cell
# ident = pd.read_csv("../data/raw/GSE133344_filtered_cell_identities.csv.gz")

ident_idx = ident.set_index("cell_barcode")
adata.obs = adata.obs.join(ident_idx, how="left")
# join in py is done by index
# adata.obs is already indexed by barcode (which is already appended with lane ID since barcodes were repeated across lanes)

print(adata.obs["guide_identity"].isna().sum(), "cells with no guide call")
print(adata.obs.shape)
adata.obs.head()

In [ ]:
# only 223 cells have no guide call; there are 111k rows (cells) - this is 0.2% - safe to just drop

adata = adata[adata.obs["guide_identity"].notna()].copy()
print(adata.shape)

In [ ]:
# look at good_coverage (metric decided by authors) and number_of_cells for more QC

print(adata.obs["good_coverage"].value_counts())
print(adata.obs["number_of_cells"].value_counts())

In [ ]:
# there are 6.9k cells with bad coverage
# the number_of_cells is correlated with sets that got the same set of CRISPR guides delivered (effectively, this is probably "cell replicates")
# so, likely the bad coverage cells are all in unique "categories" with 0 cells assigned to them
# drop the cells with bad coverage.
# adata.obs good coverage is actually an object, not true Bool values. convert first.

mask = adata.obs["good_coverage"].astype(bool).values
adata = adata[mask].copy()
print(adata.shape)

In [ ]:

# add a col to adata.var (the )
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None)

adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()

In [ ]:
print(adata.var.shape)
print(adata.var.columns.tolist())
print(adata.var)
